# NDPI Cut Extraction

Detects tissue cuts in NDPI whole-slide images with a two-step workflow:

1. Save small preview images and a `_cuts.json` manifest with level-0 cut boxes.
2. After visual approval, generate pyramidal TIFFs from the verified manifest.

All detection and I/O logic lives in
`src/data_preparation/extract_ndpi_cuts.py`.  This notebook only imports from it.


In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

PROJECT_ROOT = Path("__file__").resolve().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
from src.data_preparation.extract_ndpi_cuts import (
    create_pyramidal_tiffs_from_manifest,
    create_tissue_mask,
    detect_and_save_cut_previews,
    extract_specimen_cuts,
    open_slide_and_thumbnail,
    parse_n_cuts_from_stem,
)


## 1. Unit-test: `parse_n_cuts_from_stem`

Verify the filename → cut-count parser against known examples before running
anything that touches the NDPI files.

In [ ]:
cases = {
    "CHN_AU_10_19-21" : 3,
    "CHN_AU_10_31-33" : 3,
    "LHP_SP_6_3-4"    : 2,
    "CHN_W_7_49-50"   : 2,
    "CHN_W_2_1-3"     : 3,
    "LHP_W_17_22-24"  : 3,
}

all_ok = True
for stem, expected in cases.items():
    result = parse_n_cuts_from_stem(stem)
    status = "OK" if result == expected else "FAIL"
    if result != expected:
        all_ok = False
    print(f"  [{status}]  {stem:<30s}  expected={expected}  got={result}")

assert all_ok, "parse_n_cuts_from_stem failed for one or more cases"
print("\nAll cases passed.")

## 2. Configuration

In [ ]:
DATA_DIR   = "../data"  # change to any available data directory

NDPI_FILE  = Path(DATA_DIR) / "ndpi/CHN_AU_10_19-21.ndpi"
OUTPUT_DIR = Path(DATA_DIR) / "cuts/CHN_AU_10_19-21"   # change to any available NDPI

# NDPI_FILE  = "../sample_data/CHN_W_2_22-24.ndpi"   # change to any available NDPI
# OUTPUT_DIR = "../sample_data_out"

# Detection parameters — defaults work for most slides
PARAMS = dict(
    output_format         = "pyramidal",  # 'pyramidal' | 'tif' | 'png'
    output_levels         = None,          # None -> embed all pyramid levels
    write_manifest        = True,
    n_boxes               = None,          # None -> auto-derived from filename
    # --- tissue masking ---
    # 'saturation' (default): tissue = HSV-S >= tissue_sat_min; robust to gray backgrounds.
    # 'brightness': legacy mode for near-white backgrounds (mean RGB >= ~215).
    mask_method           = "saturation",
    tissue_sat_min        = 15,            # lower if faint tissue is missed
    blur_ksize            = 5,             # Gaussian pre-blur; 0 to disable
    background_brightness = 220,           # only used when mask_method='brightness'
    saturation_min        = 10,            # only used when mask_method='brightness'
    # --- geometry ---
    closing_frac          = 0.03,
    min_area_fraction     = 0.0001,
    margin_frac           = 0.05,
)

print(f"Input  : {NDPI_FILE}")
print(f"Output : {OUTPUT_DIR}")
print(f"Auto n_boxes: {parse_n_cuts_from_stem(Path(NDPI_FILE).stem)}")


## 3. Phase 1 — Detect cuts and save previews

This runs the full tissue-detection pipeline, writes one small preview image
per detected cut, and records level-0 bounding boxes in the manifest.


In [ ]:
preview_paths, boxes = detect_and_save_cut_previews(
    NDPI_FILE,
    OUTPUT_DIR,
    n_boxes               = PARAMS["n_boxes"],
    mask_method           = PARAMS["mask_method"],
    tissue_sat_min        = PARAMS["tissue_sat_min"],
    blur_ksize            = PARAMS["blur_ksize"],
    background_brightness = PARAMS["background_brightness"],
    saturation_min        = PARAMS["saturation_min"],
    closing_frac          = PARAMS["closing_frac"],
    min_area_fraction     = PARAMS["min_area_fraction"],
    margin_frac           = PARAMS["margin_frac"],
)

print(f"\nSaved {len(preview_paths)} preview file(s):")
for p, box in zip(preview_paths, boxes):
    print(f"  {p.name}  ({p.stat().st_size / 1_048_576:.2f} MB)  bbox={box}")


### Tissue-mask diagnostic

Displays the raw thumbnail and the tissue mask side by side so you can verify
that tissue and background are cleanly separated before inspecting cut previews.
Increase `tissue_sat_min` if background noise bleeds into the mask;
decrease it if faint tissue fragments are missing.
Switch to `mask_method='brightness'` for slides with a near-white background.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

_slide, _thumb, _, _, _ = open_slide_and_thumbnail(NDPI_FILE)
_slide.close()

_mask = create_tissue_mask(
    _thumb,
    method         = PARAMS["mask_method"],
    tissue_sat_min = PARAMS["tissue_sat_min"],
    blur_ksize     = PARAMS["blur_ksize"],
    background_brightness = PARAMS["background_brightness"],
    saturation_min = PARAMS["saturation_min"],
)

fig, (ax_img, ax_mask) = plt.subplots(1, 2, figsize=(12, 4))
ax_img.imshow(_thumb)
ax_img.set_title("Detection thumbnail (highest pyramid level)", fontsize=9)
ax_img.axis("off")
ax_mask.imshow(_mask, cmap="gray", vmin=0, vmax=1)
ax_mask.set_title(
    f"Tissue mask  (method={PARAMS['mask_method']!r}, "
    f"tissue_sat_min={PARAMS['tissue_sat_min']}, "
    f"blur_ksize={PARAMS['blur_ksize']})",
    fontsize=9,
)
ax_mask.axis("off")
plt.tight_layout()
plt.show()

tissue_frac = _mask.mean()
print(f"Tissue fraction: {tissue_frac:.1%}  "
      f"({'OK — re-run detect if mask looks wrong' if 0.01 < tissue_frac < 0.5 else 'WARNING: unexpected fraction'})"
)
del _slide, _thumb, _mask


## Visual-inspection gate

**Before running Phase 2:** inspect the previews below. Verify that each cut
contains the expected tissue region with reasonable margins. If any box is
wrong, re-run Phase 1 with adjusted parameters such as `closing_frac` or
`margin_frac`. Do not run Phase 2 until all cuts look correct.


In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np
import tifffile

stem = Path(NDPI_FILE).stem
expected_cuts = parse_n_cuts_from_stem(stem)
manifest_path = Path(OUTPUT_DIR) / f"{stem}_cuts.json"

assert len(preview_paths) == expected_cuts, (
    f"Expected {expected_cuts} preview(s), got {len(preview_paths)}"
)
for i, p in enumerate(preview_paths):
    expected_name = f"{stem}_cut{i:03d}_preview.png"
    assert p.name == expected_name, f"Expected {expected_name}, got {p.name}"
    assert p.exists() and p.stat().st_size > 0, f"{p} is missing or empty"
    assert p.stat().st_size <= 5 * 1_048_576, f"{p} is larger than 5 MB"

manifest = json.loads(manifest_path.read_text())
assert len(manifest["cuts"]) == expected_cuts
for cut, p in zip(manifest["cuts"], preview_paths):
    assert cut["preview_path"] == p.name

print(f"[OK] {len(preview_paths)} preview(s) and manifest preview paths")

fig, axes = plt.subplots(1, len(preview_paths), figsize=(5 * len(preview_paths), 4))
axes = np.atleast_1d(axes)
for ax, p in zip(axes, preview_paths):
    img = plt.imread(p)
    ax.imshow(img)
    ax.set_title(p.stem, fontsize=8)
    ax.axis("off")
plt.suptitle("Cut previews for visual approval", fontsize=11)
plt.tight_layout()
plt.show()


## 4. Phase 2 — Generate pyramidal TIFFs

Run this only after the preview images above have been inspected and approved.


In [ ]:
saved = create_pyramidal_tiffs_from_manifest(
    manifest_path,
    NDPI_FILE,
    OUTPUT_DIR,
    output_levels=PARAMS["output_levels"],
)

print(f"\nAvailable {len(saved)} pyramidal TIFF file(s):")
for p in saved:
    print(f"  {p}  ({p.stat().st_size / 1_048_576:.1f} MB)")


## 5. Validate pyramidal outputs

Checks file count, non-empty files, pyramidal structure, manifest consistency,
and idempotent `skip_existing=True` re-runs.


In [ ]:
# file count and size
assert len(saved) == expected_cuts, f"Expected {expected_cuts} cut(s), got {len(saved)}"
for p in saved:
    assert p.exists() and p.stat().st_size > 0, f"{p} is missing or empty"
print(f"[OK] {len(saved)} non-empty cut TIFF file(s)")

# manifest
manifest = json.loads(manifest_path.read_text())
assert len(manifest["cuts"]) == expected_cuts
print(
    f"[OK] Manifest — {len(manifest['cuts'])} cut(s) "
    f"mpp=({manifest['mpp_x']}, {manifest['mpp_y']}) µm/px"
)

# pyramidal structure
level0_sizes = []
for p, cut in zip(saved, manifest["cuts"]):
    with tifffile.TiffFile(str(p)) as tf:
        n_levels = len(tf.series[0].levels) if tf.series else len(tf.pages)
        level0_shape = tf.series[0].levels[0].shape[:2]
    expected_width, expected_height = cut["level0_size"]
    assert level0_shape == (expected_height, expected_width)
    level0_sizes.append(level0_shape)
    print(f"[OK] {p.name}  {n_levels} level(s)  level-0={level0_shape}")

# skip_existing=True should return the same paths without rewriting names
rerun = create_pyramidal_tiffs_from_manifest(manifest_path, NDPI_FILE, OUTPUT_DIR)
assert [p.name for p in rerun] == [p.name for p in saved]
print("[OK] skip_existing=True is idempotent")


## 6. Batch processing

Run the backward-compatible wrapper over all NDPI files in a directory. Slides
whose output directory already contains the expected number of cut TIFFs are skipped.


In [ ]:
NDPI_DIR     = Path("../data/ndpi")   # directory with downloaded NDPI files
BATCH_OUTDIR = Path("../data/cuts")   # one sub-directory per slide

ndpi_files = sorted(NDPI_DIR.glob("*.ndpi"))
print(f"Found {len(ndpi_files)} NDPI file(s) in {NDPI_DIR}\n")

results = {}  # stem -> list[Path]

for ndpi in ndpi_files:
    slide_outdir = BATCH_OUTDIR / ndpi.stem
    expected     = parse_n_cuts_from_stem(ndpi.stem)
    existing     = list(slide_outdir.glob("*_cut*.tif")) if slide_outdir.exists() else []

    if expected and len(existing) == expected:
        print(f"[SKIP] {ndpi.name}  ({expected} cut(s) already present)")
        results[ndpi.stem] = existing
        continue

    print(f"[RUN ] {ndpi.name}  (expecting {expected} cut(s))")
    results[ndpi.stem] = extract_specimen_cuts(
        str(ndpi), str(slide_outdir), **PARAMS
    )
    print()

print(f"Total cuts extracted/present: {sum(len(v) for v in results.values())}")
